# Pythonic Patterns And Anti-Patterns

## Quick notes

### What this covers
Patterns that fit Python naturally (registry via decorators and `__init_subclass__`, dependency injection, context managers, EAFP, Repository and Unit of Work, Result/Option returns, dataclass value objects) and common anti-patterns (god object, mutable default arguments, inheritance for reuse, Java-style getters/setters, singleton abuse, premature abstraction, stringly-typed code), plus a code smell cheat sheet and a before/after refactor.

### Key points
- **Registry**: a decorator adds each function to a dict, so a new handler needs no edits elsewhere. `__init_subclass__` does the same automatically for subclasses.
- **Dependency injection**: pass collaborators in (constructor or argument) instead of creating them inside. No framework needed; a `Protocol` describes the shape you expect.
- **Context manager**: `with` guarantees cleanup. Use `@contextmanager` with `try/finally`, or `__enter__`/`__exit__`.
- **EAFP** ("try it, handle the error") is preferred over **LBYP** ("check first"): shorter, and avoids check-then-use races. Catch specific exceptions.
- **Repository**: a collection-like interface (`add`, `get`, `all`) hiding where data lives. **Unit of Work**: group changes and commit or roll back together; a context manager fits.
- **Result/Option**: return `Ok(value)` or `Err(reason)` (or `Optional`) where failure is expected, and handle it with `match`.
- **Value objects**: `@dataclass(frozen=True)` gives named fields, equality by value, hashing, and immutability; use `replace()` to "change" one.
- **Anti-patterns**: god object, mutable defaults, inheritance for reuse, getters/setters, singleton abuse, premature abstraction, stringly-typed code. Each has a short fix (split, `None` default, composition, plain attribute/`@property`, pass an object in, wait for 3 cases, `Enum`).

### Gotchas (the part worth revising twice)
- **Mutable default arguments** are created once at definition time and shared by every call: `def f(x, bucket=[])` keeps growing. Use `None` and build inside (dataclasses use `field(default_factory=list)`).
- `__init_subclass__` and decorator registries run at **import time**; if a module is never imported, its plugins are never registered.
- `__exit__` returning a truthy value **swallows** the exception. Return `False`/`None` unless you mean to suppress.
- Never use a bare `except:`; it hides real bugs and catches `KeyboardInterrupt`. EAFP means catching the *specific* expected error.
- Subclassing `list` to build a stack **leaks** `insert`, `sort`, etc. Composition gives a small honest API.
- A `Singleton` class is global state in disguise. A module-level object is already a singleton in Python; better still, inject it.
- **Rule of three**: duplication is cheaper than the wrong abstraction. Abstract on the third real case, or at a boundary you must fake in tests.
- Frozen dataclasses stop attribute assignment, but a **mutable field inside** (a list) can still change.
- Don't wrap an ORM in a Repository just because the pattern exists; that is a second layer for no gain.

### When it matters
These decide whether code stays easy to test and extend. Use injection and repositories at boundaries (email, database), context managers wherever setup and cleanup pair up, `Enum` and frozen dataclasses for clear data, and run the smell checklist in code review. Treat the anti-patterns as hints to look closer, not as laws.

### One-line recap example
```python
def add_item(item, bucket=None):      # not bucket=[]
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket
```

In [1]:
from contextlib import contextmanager
from dataclasses import dataclass, field, replace
from enum import Enum
from typing import Callable, Generic, Optional, Protocol, TypeVar

## 01. Pythonic Patterns And Anti-Patterns - Introduction

- Good Python code uses the language's own tools (decorators, context
  managers, dataclasses, duck typing) instead of copying Java-style designs.
- Part 1 (sections 02-10): patterns that fit Python well.
- Part 2 (sections 11-17): anti-patterns - common habits that cause pain.
- Part 3 (sections 18-19): a code smell cheat sheet and a before/after refactor.
- Each section: the problem, when NOT to use it, and a small demo.

## 02. Registry via decorators

- Problem: you have a growing list of handlers/commands and a big
  if/elif that must be edited for every new one.
- Idea: a decorator adds each function to a dict at definition time.
- New handler = new function with a decorator. Nothing else changes.
- When NOT to use: two or three fixed handlers - a plain dict is fine.
  Also note a registry only fills when its module is imported.

In [2]:
COMMANDS = {}

def command(name):
    def register(fn):
        COMMANDS[name] = fn
        return fn                      # return fn so it stays usable
    return register

@command("greet")
def greet(who):
    return f"hello {who}"

@command("shout")
def shout(who):
    return f"HEY {who.upper()}"

def run(name, arg):
    fn = COMMANDS.get(name)
    if fn is None:
        return f"unknown command: {name}"
    return fn(arg)

print(run("greet", "ana"), "|", run("shout", "ana"), "|", run("nope", "ana"))
print(sorted(COMMANDS))

hello ana | HEY ANA | unknown command: nope
['greet', 'shout']


## 03. Plugin registry via __init_subclass__

- Problem: you want every subclass of a base to register itself
  automatically, with no decorator and no metaclass.
- Idea: __init_subclass__ runs once whenever a subclass is defined.
- Class keyword arguments (class X(Base, kind="x")) pass config in.
- When NOT to use: if registering explicitly is clearer. Magic that runs
  at import time surprises readers.

In [3]:
class Exporter:
    registry = {}
    def __init_subclass__(cls, fmt=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if fmt:
            Exporter.registry[fmt] = cls
    def export(self, data):
        raise NotImplementedError

class CsvExporter(Exporter, fmt="csv"):
    def export(self, data):
        return ",".join(map(str, data))

class PipeExporter(Exporter, fmt="pipe"):
    def export(self, data):
        return "|".join(map(str, data))

def export(fmt, data):
    return Exporter.registry[fmt]().export(data)

print(export("csv", [1, 2, 3]), export("pipe", [1, 2, 3]))
print(sorted(Exporter.registry))

1,2,3 1|2|3
['csv', 'pipe']


## 04. Dependency injection without frameworks

- Problem: a class creates its own helpers (database, mailer), so you
  can't swap them or test without the real thing.
- Idea: pass the helpers in (constructor or function argument).
- No framework needed. Use a Protocol to describe what you need.
- When NOT to use: tiny scripts, or objects that are never swapped.
  Don't inject every little thing.

In [4]:
class Mailer(Protocol):
    def send(self, to: str, body: str) -> None: ...

class SmtpMailer:
    def send(self, to, body):
        print(f"SMTP -> {to}: {body}")

class FakeMailer:                      # test double
    def __init__(self):
        self.sent = []
    def send(self, to, body):
        self.sent.append((to, body))

class SignupService:
    def __init__(self, mailer: Mailer):
        self.mailer = mailer           # injected, not created here
    def register(self, email):
        self.mailer.send(email, "welcome!")
        return f"registered {email}"

print(SignupService(SmtpMailer()).register("a@x.com"))
fake = FakeMailer()
SignupService(fake).register("b@x.com")
print(fake.sent)

# Functions can be injected too
def greet_user(name, formatter=str.title):
    return formatter(name)

print(greet_user("ana bo"), greet_user("ana bo", str.upper))

SMTP -> a@x.com: welcome!
registered a@x.com
[('b@x.com', 'welcome!')]
Ana Bo ANA BO


## 05. Context manager pattern

- Problem: setup and cleanup must always happen as a pair
  (open/close, lock/unlock, begin/rollback), even on errors.
- Idea: "with" guarantees cleanup. Write one with @contextmanager
  (generator) or with __enter__/__exit__.
- When NOT to use: no cleanup needed, or cleanup is not tied to a block.

In [5]:
@contextmanager
def tag(name):
    print(f"<{name}>")
    try:
        yield name
    finally:
        print(f"</{name}>")           # always runs, even on error

with tag("div"):
    print("inside")

try:
    with tag("p"):
        raise ValueError("boom")
except ValueError as e:
    print("caught:", e)

class Timer:                           # class form; no real clock for stable output
    def __enter__(self):
        print("enter")
        return self
    def __exit__(self, exc_type, exc, tb):
        print("exit, error:", exc_type.__name__ if exc_type else None)
        return False                   # False = do not swallow the error

with Timer():
    pass

<div>
inside
</div>
<p>
</p>
caught: boom
enter
exit, error: None


## 06. EAFP vs LBYP

- LBYP = Look Before You Leap: check first, then act.
- EAFP = Easier to Ask Forgiveness than Permission: act, handle errors.
- Python prefers EAFP: shorter, and avoids the gap between "check" and
  "use" (race conditions on files, for example).
- When NOT to use EAFP: when failure is the common case and exceptions
  are slow, or when the failing step has side effects you can't undo.
  Also never use a bare "except:" - catch the specific error.

In [6]:
data = {"name": "ana"}

# LBYP
if "age" in data:
    age = data["age"]
else:
    age = "unknown"
print("LBYP:", age)

# EAFP
try:
    age = data["age"]
except KeyError:
    age = "unknown"
print("EAFP:", age)

# Often there's an even shorter way
print("get:", data.get("age", "unknown"))

def to_int(text):
    try:
        return int(text)
    except ValueError:
        return None

print(to_int("42"), to_int("4x2"))

class Duck:
    def quack(self):
        return "quack"

def make_noise(thing):
    try:
        return thing.quack()           # duck typing: just try it
    except AttributeError:
        return "(silence)"

print(make_noise(Duck()), make_noise(42))

LBYP: unknown
EAFP: unknown
get: unknown
42 None
quack (silence)


## 07. Repository (lite)

- Problem: business code is mixed with storage code (SQL, files), so it
  can't be tested or moved to another store.
- Idea: a repository looks like a collection (add, get, list) and hides
  where the data lives.
- Here: in-memory version. A SQL version would have the same methods.
- When NOT to use: a small script or CRUD app where the ORM already is
  the abstraction. Don't wrap an ORM in a second ORM.

In [7]:
@dataclass
class User:
    id: int
    name: str

class InMemoryUserRepo:
    def __init__(self):
        self._rows = {}
    def add(self, user):
        self._rows[user.id] = user
    def get(self, user_id):
        return self._rows.get(user_id)
    def all(self):
        return [self._rows[k] for k in sorted(self._rows)]

def rename(repo, user_id, new_name):  # business code only knows the repo API
    user = repo.get(user_id)
    if user is None:
        return "no such user"
    user.name = new_name
    return "renamed"

repo = InMemoryUserRepo()
repo.add(User(2, "bo"))
repo.add(User(1, "ana"))
print(rename(repo, 1, "anna"), rename(repo, 9, "x"))
print(repo.all())

renamed no such user
[User(id=1, name='anna'), User(id=2, name='bo')]


## 08. Unit of Work (lite)

- Problem: several changes must succeed together or not at all.
- Idea: collect changes, then commit once; on error roll back.
- Python: a context manager is a natural Unit of Work.
- When NOT to use: a real database transaction already gives you this.
  This is just the idea in miniature.

In [8]:
class UnitOfWork:
    def __init__(self, store):
        self.store = store             # a plain dict
        self._pending = {}
    def set(self, key, value):
        self._pending[key] = value
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            self.store.update(self._pending)   # commit
            print("committed", sorted(self._pending))
        else:
            print("rolled back", sorted(self._pending))
        return False

db = {}
with UnitOfWork(db) as uow:
    uow.set("a", 1)
    uow.set("b", 2)
print(db)

try:
    with UnitOfWork(db) as uow:
        uow.set("c", 3)
        raise RuntimeError("fail midway")
except RuntimeError:
    pass
print(db)                              # "c" never saved

committed ['a', 'b']
{'a': 1, 'b': 2}
rolled back ['c']
{'a': 1, 'b': 2}


## 09. Result / Option-style returns

- Problem: functions that fail either raise (surprise!) or return None
  or -1 (easy to forget to check).
- Idea: return a small object that says "ok with value" or "error with
  reason", so the caller must look at it.
- Optional[T] (value or None) is the "Option" half.
- When NOT to use: Python's normal answer is exceptions. Use Result only
  at boundaries where failure is expected (parsing user input, validation).

In [9]:
T = TypeVar("T")

@dataclass(frozen=True)
class Ok(Generic[T]):
    value: T

@dataclass(frozen=True)
class Err:
    reason: str

def parse_age(text) -> "Ok[int] | Err":
    if not text.isdigit():
        return Err(f"not a number: {text!r}")
    age = int(text)
    if age > 150:
        return Err("too old")
    return Ok(age)

for raw in ["30", "abc", "200"]:
    match parse_age(raw):
        case Ok(value=v):
            print("ok", v)
        case Err(reason=r):
            print("error:", r)

def find_user(users, name) -> Optional[str]:
    return next((u for u in users if u == name), None)

found = find_user(["ana", "bo"], "cy")
print("found:", found if found is not None else "nobody")

ok 30
error: not a number: 'abc'
error: too old
found: nobody


## 10. Dataclass value objects

- Problem: passing around loose tuples/dicts (x, y) or ("USD", 5) is
  unclear and easy to mix up.
- Idea: a small frozen dataclass: named fields, equality by value,
  hashable, cannot be changed by accident.
- Change = make a new copy with dataclasses.replace.
- When NOT to use: objects with identity and a changing life (a User,
  an Order) - those are entities, not value objects.

In [10]:
@dataclass(frozen=True, order=True)
class Money:
    amount: int                        # cents, to avoid float errors
    currency: str = "USD"
    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("currency mismatch")
        return Money(self.amount + other.amount, self.currency)

a, b = Money(500), Money(250)
print(a + b)
print(a == Money(500), a < Money(600))
print({a: "five dollars"}[Money(500)])      # hashable, usable as dict key
bigger = replace(a, amount=900)
print(a, bigger)
try:
    a.amount = 1
except Exception as e:
    print(type(e).__name__)
try:
    a + Money(1, "EUR")
except ValueError as e:
    print("ValueError:", e)

Money(amount=750, currency='USD')
True True
five dollars
Money(amount=500, currency='USD') Money(amount=900, currency='USD')
FrozenInstanceError
ValueError: currency mismatch


## 11. Anti-pattern: God Object

- Problem: one huge class knows and does everything (users, emails,
  reports, storage). Every change touches it; nothing can be tested alone.
- Fix: split by responsibility (see SOLID: Single Responsibility).
- Smell: class name ends in Manager/Handler/Utils and has 20+ methods.
- When it's OK: a short script. Don't split what is only 15 lines.

In [11]:
class AppManagerBad:                   # does too much
    def __init__(self):
        self.users = []
    def add_user(self, name):
        self.users.append(name)
    def send_email(self, name):
        return f"mail to {name}"
    def build_report(self):
        return f"{len(self.users)} users"
    def save_to_disk(self):
        return "saved"

class UserBook:
    def __init__(self):
        self.users = []
    def add(self, name):
        self.users.append(name)

class EmailService:
    def send(self, name):
        return f"mail to {name}"

def user_report(book):
    return f"{len(book.users)} users"

book = UserBook()
book.add("ana")
print(EmailService().send("ana"), "|", user_report(book))

mail to ana | 1 users


## 12. Anti-pattern: Mutable default arguments

- Problem: a default value like [] or {} is created ONCE when the
  function is defined, then shared by every call.
- Fix: default to None and create the new object inside.
- When it's OK: immutable defaults (numbers, strings, tuples, None).

In [12]:
def add_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item_bad("a"))
print(add_item_bad("b"))               # surprise: still has "a"

def add_item(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item("a"))
print(add_item("b"))

@dataclass
class Cart:
    items: list = field(default_factory=list)   # dataclass guards this for you

c1, c2 = Cart(), Cart()
c1.items.append("x")
print(c1.items, c2.items)

['a']
['a', 'b']
['a']
['b']
['x'] []


## 13. Anti-pattern: Over-using inheritance

- Problem: inheriting just to reuse code creates tight coupling and
  weird "is-a" lies (a Stack is not a list).
- Fix: composition ("has-a"): hold the helper as an attribute and
  expose only what you want.
- Test: can you say "X is a kind of Y" truthfully in every case?
- When inheritance IS fine: real is-a relationships, ABCs, mixins with a
  clear single job.

In [13]:
class StackBad(list):                  # exposes insert(), sort(), etc.
    def push(self, x):
        self.append(x)

s_bad = StackBad()
s_bad.push(1)
s_bad.push(2)
s_bad.insert(0, 99)                    # breaks stack rules!
print(list(s_bad))

class Stack:                           # composition: small, honest API
    def __init__(self):
        self._items = []
    def push(self, x):
        self._items.append(x)
    def pop(self):
        return self._items.pop()
    def __len__(self):
        return len(self._items)

s = Stack()
s.push(1)
s.push(2)
print(s.pop(), len(s), hasattr(s, "insert"))

[99, 1, 2]
2 1 False


## 14. Anti-pattern: Java-style getters and setters

- Problem: get_x()/set_x() for every field is noise. Python attributes
  are public by default and can be upgraded later.
- Fix: use a plain attribute. If you later need validation, switch to
  @property - callers don't change.
- When NOT to avoid: a property is right when you must validate,
  compute, or keep two fields in sync.

In [14]:
class PersonBad:
    def __init__(self, name):
        self._name = name
    def get_name(self):
        return self._name
    def set_name(self, v):
        self._name = v

class Person:
    def __init__(self, name):
        self.name = name               # plain attribute

class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius         # goes through the setter below
    @property
    def celsius(self):
        return self._celsius
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value
    @property
    def fahrenheit(self):              # computed, read-only
        return self._celsius * 9 / 5 + 32

p = Person("ana")
p.name = "anna"
print(p.name)
t = Temperature(100)
print(t.celsius, t.fahrenheit)
try:
    t.celsius = -500
except ValueError as e:
    print("ValueError:", e)

anna
100 212.0
ValueError: below absolute zero


## 15. Anti-pattern: Singleton abuse

- Problem: singletons are global state in disguise: hidden dependencies,
  tests that affect each other, hard to replace.
- Python already gives you one: a module is created once per process.
  So a module-level object IS a singleton, with no class tricks.
- Better still: create one object and inject it (section 04).
- When it's OK: truly process-wide resources (a logger, a config loaded once).

In [15]:
class SettingsBad:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.debug = False
        return cls._instance

x, y = SettingsBad(), SettingsBad()
x.debug = True
print("y.debug:", y.debug, "(changed through x!)")

# Module-level object style: create one, pass it around
@dataclass
class Settings:
    debug: bool = False

def run_app(settings):
    return "debug mode" if settings.debug else "normal mode"

print(run_app(Settings()), "|", run_app(Settings(debug=True)))   # each test gets its own

y.debug: True (changed through x!)
normal mode | debug mode


## 16. Anti-pattern: Premature abstraction

- Problem: building interfaces, factories, and layers for variations that
  do not exist yet. More code, more places for bugs, harder to read.
- Rule of three: copy once, copy twice, abstract on the third.
  Wrong abstraction costs more than duplication.
- YAGNI = You Aren't Gonna Need It.
- When abstraction IS right: you already have 2-3 real variants, or the
  boundary is an external system you must be able to swap or fake.

In [16]:
class GreeterFactoryBad:               # 3 classes to say hello
    def create(self):
        return GreeterBad()

class GreeterBad:
    def greet(self, name):
        return f"hello {name}"

print(GreeterFactoryBad().create().greet("ana"))

def hello(name):                       # same result
    return f"hello {name}"

print(hello("ana"))

hello ana
hello ana


## 17. Anti-pattern: Stringly-typed code

- Problem: using raw strings ("pending", "shipped") for fixed choices.
  One typo ("shiped") silently breaks logic; tools can't help.
- Fix: Enum for fixed choices; dataclass or TypedDict for records
  instead of dicts with magic keys.
- When strings are fine: free text, or data crossing a boundary
  (JSON) - convert to Enum at the edge.

In [17]:
def can_cancel_bad(status):
    return status == "pending" or status == "paid"

print(can_cancel_bad("shiped"), "<- typo, no error, wrong answer")

class Status(Enum):
    PENDING = "pending"
    PAID = "paid"
    SHIPPED = "shipped"

def can_cancel(status: Status):
    return status in (Status.PENDING, Status.PAID)

print(can_cancel(Status.SHIPPED), can_cancel(Status("pending")))
try:
    Status("shiped")                   # typo fails loudly at the edge
except ValueError as e:
    print("ValueError:", e)

False <- typo, no error, wrong answer
False True
ValueError: 'shiped' is not a valid Status


## 18. Code smell cheat sheet

- A smell is a hint, not a bug: look closer, don't panic.
- Long function ........ split into small named functions
- Big class ............ split by responsibility (god object)
- Deep nesting ......... early return / guard clauses
- Duplicate code ....... extract a function (after the 3rd copy)
- Many parameters ...... group into a dataclass
- Magic numbers ........ named constants / Enum
- Boolean flag params .. two functions, or an Enum
- Bare except .......... catch the specific error
- Mutable defaults ..... None + create inside
- Comments explaining bad names .. rename instead
- isinstance chains .... polymorphism, singledispatch, match
- Global state ......... pass it in (dependency injection)

In [18]:
SMELLS = [
    ("Long function", "split into named helpers"),
    ("Deep nesting", "guard clauses / early return"),
    ("Many parameters", "group into a dataclass"),
    ("Magic numbers", "named constants or Enum"),
    ("Boolean flag argument", "two functions or an Enum"),
    ("Bare except", "catch the specific error"),
    ("isinstance chain", "singledispatch / match / polymorphism"),
    ("Global state", "inject it"),
]
for smell, fix in SMELLS:
    print(f"{smell:22} => {fix}")

Long function          => split into named helpers
Deep nesting           => guard clauses / early return
Many parameters        => group into a dataclass
Magic numbers          => named constants or Enum
Boolean flag argument  => two functions or an Enum
Bare except            => catch the specific error
isinstance chain       => singledispatch / match / polymorphism
Global state           => inject it


## 19. Refactor: before and after

- Take one messy function and apply the smells above step by step.
- Behavior stays the same; only the shape improves.
- Always keep a quick check (here: assert) so you know nothing broke.

In [19]:
def process_bad(o, t):                 # unclear names, magic numbers, nesting
    if o is not None:
        if len(o) > 0:
            tot = 0
            for i in o:
                tot = tot + i["p"] * i["q"]
            if t == "vip":
                tot = tot * 0.8
            elif t == "new":
                tot = tot * 0.95
            if tot > 100:
                tot = tot - 10
            return tot
        else:
            return 0
    else:
        return 0

class Tier(Enum):
    VIP = 0.80
    NEW = 0.95
    REGULAR = 1.0

BULK_THRESHOLD = 100
BULK_BONUS = 10

@dataclass(frozen=True)
class Line:
    price: float
    qty: int

def subtotal(lines):
    return sum(line.price * line.qty for line in lines)

def order_total(lines, tier=Tier.REGULAR):
    if not lines:                      # guard clause replaces nesting
        return 0
    total = subtotal(lines) * tier.value
    if total > BULK_THRESHOLD:
        total -= BULK_BONUS
    return total

old_items = [{"p": 50, "q": 2}, {"p": 20, "q": 1}]
new_items = [Line(50, 2), Line(20, 1)]
before = process_bad(old_items, "vip")
after = order_total(new_items, Tier.VIP)
print("before:", before, "| after:", after)
assert abs(before - after) < 1e-9
assert process_bad([], "vip") == order_total([], Tier.VIP) == 0
print("same behavior, clearer code")

before: 96.0 | after: 96.0
same behavior, clearer code
